In [ ]:
TEST_WEEK = 2

####### LOAD 
from pathlib import Path

from shared.course_schedule import build_all_course_schedules
from shared.markdown import * 

OUTPUT_DIR = Path("outputs/weekly_briefs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


all_schedules = build_all_course_schedules(
    sections_file="data/f2026_sections.csv",
    terms_file="data/terms.csv"
)

courses = {}

for schedule in all_schedules:

    course_code = schedule["course_code"]

    if course_code not in courses:
        courses[course_code] = {
            "course_name": schedule["course_name"],
            "schedules": []
        }

    courses[course_code]["schedules"].append(schedule)

def add_list_or_text(lines, value):
    if isinstance(value, list):
        for item in value:
            lines.append(f"- {item}")
    elif value:
        lines.append(str(value))

def build_weekly_dashboard(courses, week_num):
    lines = []

    first_schedule = next(iter(courses.values()))["schedules"][0]
    first_week = first_schedule["weeks"][week_num - 1]

    week_label = first_week.get("week")
    if week_label is None:
        lines.append(f"## {first_week.get('topic', 'Scheduled Event')}")
    else:
        lines.append(f"## Week {week_label}")

    lines.append(first_week["date_str"])
    lines.append("")

    for course in courses.values():

        master = course["schedules"][0]
        week = master["weeks"][week_num - 1]

        readings = week.get("readings", [])
        assignments = week.get("assignments", [])
        lab = week.get("lab")
        notes = week.get("notes")

        lines.append(f"# {master['course_code']}: {master['course_name']}")
        lines.append("")

        add_heading(lines, "Topic")
        lines.append(week.get("topic", ""))

        if notes:
            lines.append(f"**[Lecture Notes]({notes})**")

        lines.append("")

        add_heading(lines, "Readings")
        add_list_or_text(lines, readings)
        lines.append("")

        if assignments:
            add_heading(lines, "Student Deliverables")
            add_list_or_text(lines, assignments)
            lines.append("")

        if lab:
            add_heading(lines, "Lab")
            lines.append(lab)
            lines.append("")

        add_heading(lines, "Teaching Schedule")

        for schedule in course["schedules"]:
            lines.append(
                f"- Section {schedule['section']} "
                f"({schedule.get('class_time', 'Time TBD')})"
            )

        lines.append("")

        add_heading(lines, "Prep Checklist")

        lines.append("- [ ] Update slides")
        lines.append("- [ ] Update poll")
        lines.append("- [ ] Update speaking notes")
        lines.append("- [ ] Upload slides to OWL")
        lines.append("- [ ] Post announcement")
        lines.append("---")

    return "\n".join(lines)


dashboard = build_weekly_dashboard(courses, TEST_WEEK)

output_file = OUTPUT_DIR / f"week_{TEST_WEEK}_dashboard.md"

with open(output_file, "w", encoding="utf-8") as f:
    f.write(dashboard)

print(output_file)

outputs\weekly_briefs\week_2_dashboard.md
